# Filter Explorer

Analyzes and compares digital filter frequency responses for two signals in real time.
Plots magnitude (dB) and phase (°) on a logarithmic frequency axis (20 Hz – 20 kHz).

| Filter type | Notes |
|-------------|-------|
| **Butterworth** | Maximally flat magnitude, no ripple |
| **Chebyshev I** | Equiripple in passband, steeper roll-off than Butterworth |
| **Chebyshev II** | Equiripple in stopband, flat passband |
| **Linkwitz-Riley** | Butterworth² — standard crossover filter, −6 dB at Fc, sums to flat |
| **All-Pass 1st / 2nd order** | Unity magnitude, phase-only shaping (alignment / time correction) |

**Input signals:** unit impulse (ideal) or any mono/stereo `.wav` file.  
**Controls per channel:** gain (±30 dB), polarity (0° / 180°), delay (ms), filter type, order, cutoff, pass type.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter, cheby1, cheby2
import tkinter as tk
from tkinter import ttk, filedialog
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import scipy.io.wavfile as wavfile

# ─── Configuración global ─────────────────────────────────────────────────────
N  = 4096   # Número de muestras para la respuesta al impulso
fs = 48000  # Frecuencia de muestreo por defecto (Hz)

es_ideal_onda1 = True
es_ideal_onda2 = True
onda1_original = None
onda2_original = None
fs1 = fs
fs2 = fs
# ─────────────────────────────────────────────────────────────────────────────


def all_pass_1st_order(cutoff, fs):
    """All-pass de primer orden: fase lineal de 0° a −180°."""
    omega = 2 * np.pi * cutoff / fs
    alpha = (1 - np.sin(omega)) / np.cos(omega)
    return [alpha, -1], [1, -alpha]


def all_pass_2nd_order(cutoff, fs, Q=0.707):
    """All-pass de segundo orden: fase de 0° a −360°, Q controla la pendiente."""
    omega = 2 * np.pi * cutoff / fs
    alpha = np.sin(omega) / (2 * Q)
    a0 = 1 + alpha
    a1 = -2 * np.cos(omega)
    a2 = 1 - alpha
    b = [a2 / a0, a1 / a0, a0 / a0]
    a = [1,       a1 / a0, a2 / a0]
    return b, a


def seleccionar_filtro(tipo_filtro, order, cutoff, fs, btype, ripple=None, bandwidth=None):
    nyq = 0.5 * fs
    wn  = cutoff / nyq
    if tipo_filtro == "Butterworth":
        return butter(order, wn, btype=btype)
    elif tipo_filtro == "Chebyshev 1":
        return cheby1(order, ripple or 1, wn, btype=btype)
    elif tipo_filtro == "Chebyshev 2":
        return cheby2(order, ripple or 40, wn, btype=btype)
    elif tipo_filtro == "Linkwitz-Riley":
        # LR = Butterworth² → aplicar dos veces (aquí se devuelven coefs del sub-filtro)
        return butter(max(order // 2, 1), wn, btype=btype)
    elif tipo_filtro == "All Pass":
        if order == 1:
            return all_pass_1st_order(cutoff, fs)
        else:
            return all_pass_2nd_order(cutoff, fs, bandwidth or 0.707)
    raise ValueError(f"Tipo de filtro desconocido: {tipo_filtro}")


def aplicar_filtro(signal, tipo_filtro, order, cutoff, fs, btype, ripple, bandwidth):
    if tipo_filtro == "none":
        return signal
    b, a = seleccionar_filtro(tipo_filtro, order, cutoff, fs, btype, ripple, bandwidth)
    filtered = lfilter(b, a, signal)
    if tipo_filtro == "Linkwitz-Riley":   # Aplicar dos veces para LR
        filtered = lfilter(b, a, filtered)
    return filtered


def aplicar_delay(signal, fs, delay_ms):
    n = int(delay_ms * fs / 1000)
    if n <= 0:
        return signal
    out = np.zeros_like(signal)
    if n < len(signal):
        out[n:] = signal[:-n]
    return out


def safe_float(entry, default, lo=None, hi=None):
    try:
        v = float(entry.get())
        if lo is not None and v < lo: return default
        if hi is not None and v > hi: return default
        return v
    except ValueError:
        return default


def safe_int(entry, default, lo=1):
    try:
        v = int(entry.get())
        return v if v >= lo else default
    except ValueError:
        return default


def cargar_wav(canal):
    global es_ideal_onda1, es_ideal_onda2, onda1_original, onda2_original, fs1, fs2
    path = filedialog.askopenfilename(filetypes=[("Archivos WAV", "*.wav")])
    if not path:
        return
    try:
        rate, data = wavfile.read(path)
        data = data[:, 0] if data.ndim == 2 else data  # estéreo → canal izquierdo
        # Ajustar longitud a N
        if len(data) > N:
            data = data[:N]
        else:
            data = np.pad(data, (0, N - len(data)))
        data = data.astype(np.float64)
        if canal == 1:
            onda1_original, fs1, es_ideal_onda1 = data, rate, False
        else:
            onda2_original, fs2, es_ideal_onda2 = data, rate, False
        actualizar_señales()
    except Exception as e:
        error_label.config(text=f"Error: {e}")


def actualizar_señales(*_):
    error_label.config(text="")

    fs_1 = fs1 if not es_ideal_onda1 else fs
    fs_2 = fs2 if not es_ideal_onda2 else fs

    g1 = ganancia1_scale.get()
    g2 = ganancia2_scale.get()
    pol1 = 0.0 if polaridad1_var.get() == "0" else np.pi
    pol2 = 0.0 if polaridad2_var.get() == "0" else np.pi

    # Construir señales de entrada
    if es_ideal_onda1:
        s1 = np.zeros(N); s1[0] = 10 ** (g1 / 20) * np.cos(pol1)
    else:
        s1 = onda1_original.copy() * 10 ** (g1 / 20) * np.cos(pol1) if pol1 == 0 else -onda1_original.copy() * 10 ** (g1 / 20)

    if es_ideal_onda2:
        s2 = np.zeros(N); s2[0] = 10 ** (g2 / 20) * np.cos(pol2)
    else:
        s2 = onda2_original.copy() * 10 ** (g2 / 20) * np.cos(pol2) if pol2 == 0 else -onda2_original.copy() * 10 ** (g2 / 20)

    # Aplicar filtros
    s1 = aplicar_filtro(s1, filtro1_var.get(), safe_int(orden1_entry, 1),
                        safe_float(cutoff1_entry, 1000, 20, 20000), fs_1,
                        tipo1_var.get(), safe_float(riple_entry, 1),
                        safe_float(bandwidth_entry, 0.707))
    s2 = aplicar_filtro(s2, filtro2_var.get(), safe_int(orden2_entry, 1),
                        safe_float(cutoff2_entry, 1000, 20, 20000), fs_2,
                        tipo2_var.get(), safe_float(riple_entry, 1),
                        safe_float(bandwidth_entry, 0.707))

    # Aplicar delay
    s1 = aplicar_delay(s1, fs_1, safe_float(delay1_text, 0, lo=0))
    s2 = aplicar_delay(s2, fs_2, safe_float(delay2_text, 0, lo=0))

    # FFT
    eps   = 1e-10
    freqs = np.fft.fftfreq(N, 1 / fs_1)
    idx   = np.where((freqs >= 20) & (freqs <= 20000))

    def mag_db(s): return 20 * np.log10(np.abs(np.fft.fft(s)) + eps)
    def phase_deg(s): return np.angle(np.fft.fft(s)) * 180 / np.pi

    xticks     = [31.5, 63, 125, 250, 500, 1000, 4000, 8000, 16000]
    xtick_str  = ['31.5','63','125','250','500','1k','4k','8k','16k']

    ax1.clear()
    for sig, lbl in [(s1,'Onda 1'),(s2,'Onda 2'),(s1+s2,'Suma'),((s1+s2)/2,'Promedio')]:
        ax1.semilogx(freqs[idx], mag_db(sig)[idx], label=lbl)
    ax1.axhline(-3, color='black', linestyle='--', linewidth=0.7, label='-3 dB')
    ax1.set(xlabel='Frecuencia (Hz)', ylabel='Magnitud (dB)',
            title='Respuesta en Magnitud', xlim=[20, 20000])
    ax1.set_xticks(xticks); ax1.set_xticklabels(xtick_str)
    ax1.legend(fontsize=8); ax1.grid(True)

    ax2.clear()
    for sig, lbl in [(s1,'Onda 1'),(s2,'Onda 2'),(s1+s2,'Suma'),((s1+s2)/2,'Promedio')]:
        ax2.semilogx(freqs[idx], phase_deg(sig)[idx], label=lbl)
    ax2.set(xlabel='Frecuencia (Hz)', ylabel='Fase (°)',
            title='Respuesta en Fase', xlim=[20, 20000],
            ylim=[-185, 185], yticks=range(-180, 181, 45))
    ax2.set_xticks(xticks); ax2.set_xticklabels(xtick_str)
    ax2.legend(fontsize=8); ax2.grid(True)

    fig.tight_layout()
    canvas.draw()


# ─── GUI ──────────────────────────────────────────────────────────────────────
root = tk.Tk()
root.title("Filter Explorer — Magnitude & Phase Analyzer")
root.geometry("1440x900")

style = ttk.Style()
style.configure("TCombobox", foreground="black")

fig = Figure(figsize=(9, 7))
ax1 = fig.add_subplot(2, 1, 1)
ax2 = fig.add_subplot(2, 1, 2)
fig.tight_layout()

canvas = FigureCanvasTkAgg(fig, master=root)
canvas.draw()
canvas.get_tk_widget().grid(row=0, column=2, rowspan=20, padx=10, pady=10, sticky="nsew")

def row(label, col=0, r=0, **kw):
    tk.Label(root, text=label).grid(row=r, column=col, sticky="w", padx=10)

controls = [
    ("Ganancia Onda 1 (dB)",  "scale",    "ganancia1_scale",  dict(from_=-30, to=30, orient="horizontal")),
    ("Ganancia Onda 2 (dB)",  "scale",    "ganancia2_scale",  dict(from_=-30, to=30, orient="horizontal")),
    ("Polaridad Onda 1",      "combobox", "polaridad1_var",   dict(values=["0","180°"])),
    ("Polaridad Onda 2",      "combobox", "polaridad2_var",   dict(values=["0","180°"])),
    ("Filtro Onda 1",         "combobox", "filtro1_var",      dict(values=["none","Butterworth","Chebyshev 1","Chebyshev 2","Linkwitz-Riley","All Pass"])),
    ("Orden Filtro Onda 1",   "entry",    "orden1_entry",     dict(default="1")),
    ("Fc Onda 1 (Hz)",        "entry",    "cutoff1_entry",    dict(default="1000")),
    ("Paso Onda 1",           "combobox", "tipo1_var",        dict(values=["none","lowpass","highpass"])),
    ("Filtro Onda 2",         "combobox", "filtro2_var",      dict(values=["none","Butterworth","Chebyshev 1","Chebyshev 2","Linkwitz-Riley","All Pass"])),
    ("Orden Filtro Onda 2",   "entry",    "orden2_entry",     dict(default="1")),
    ("Fc Onda 2 (Hz)",        "entry",    "cutoff2_entry",    dict(default="1000")),
    ("Paso Onda 2",           "combobox", "tipo2_var",        dict(values=["none","lowpass","highpass"])),
    ("Delay Onda 1 (ms)",     "entry",    "delay1_text",      dict(default="0")),
    ("Delay Onda 2 (ms)",     "entry",    "delay2_text",      dict(default="0")),
    ("Ancho de banda (Q)",    "entry",    "bandwidth_entry",  dict(default="0.707")),
    ("Ripple (dB)",           "entry",    "riple_entry",      dict(default="1")),
]

widgets = {}
for r, (label, wtype, name, opts) in enumerate(controls):
    tk.Label(root, text=label).grid(row=r, column=0, sticky="w", padx=10)
    if wtype == "scale":
        w = tk.Scale(root, **opts)
        w.set(0)
        w.bind("<ButtonRelease-1>", actualizar_señales)
    elif wtype == "combobox":
        var = tk.StringVar(value=opts["values"][0])
        w = ttk.Combobox(root, textvariable=var, values=opts["values"])
        var.trace("w", actualizar_señales)
        widgets[name] = var      # store the StringVar
    else:  # entry
        w = tk.Entry(root)
        w.insert(0, opts["default"])
        w.bind("<KeyRelease>", actualizar_señales)
    w.grid(row=r, column=1, padx=10, pady=2)
    widgets.setdefault(name, w)

# Extract named widgets
ganancia1_scale = widgets["ganancia1_scale"]
ganancia2_scale = widgets["ganancia2_scale"]
polaridad1_var  = widgets["polaridad1_var"]
polaridad2_var  = widgets["polaridad2_var"]
filtro1_var     = widgets["filtro1_var"]
filtro2_var     = widgets["filtro2_var"]
tipo1_var       = widgets["tipo1_var"]
tipo2_var       = widgets["tipo2_var"]
orden1_entry    = widgets["orden1_entry"]
orden2_entry    = widgets["orden2_entry"]
cutoff1_entry   = widgets["cutoff1_entry"]
cutoff2_entry   = widgets["cutoff2_entry"]
delay1_text     = widgets["delay1_text"]
delay2_text     = widgets["delay2_text"]
bandwidth_entry = widgets["bandwidth_entry"]
riple_entry     = widgets["riple_entry"]

r_next = len(controls)
tk.Label(root, text="WAV Onda 1").grid(row=r_next,   column=0, sticky="w", padx=10)
tk.Button(root, text="Cargar WAV", command=lambda: cargar_wav(1)).grid(row=r_next,   column=1, padx=10, pady=4)
tk.Label(root, text="WAV Onda 2").grid(row=r_next+1, column=0, sticky="w", padx=10)
tk.Button(root, text="Cargar WAV", command=lambda: cargar_wav(2)).grid(row=r_next+1, column=1, padx=10, pady=4)

error_label = tk.Label(root, text="", fg="red")
error_label.grid(row=r_next+2, column=0, columnspan=2)

actualizar_señales()
root.mainloop()
